In [ ]:
"""
fitters_test.ipynb

Tests the fitter object, meant to 
fit all models to a given session.

Author: Stellina X. Ao
Created: 2026-03-05
Last Modified: 2026-03-11
Python Version: 3.11.14
"""

from sg.fitter import LVMFamily
from src.squiggs.neuron_viewer import NeuronViewer
from src.squiggs.renderers import FitRenderer
from sg.eval_models import plot_summary

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
"""
TODO:
backburner
- what about adding ReLU to the taskvar model and removing block?
- plot rasters, sanity check the psths
"""

## Fit 

In [ ]:
# load data

trial_data_all = np.load(
    "../vars/trial_data_all_MM012_MM013_all5.npz", allow_pickle=True
)["arr_0"]
session_data_all = np.load(
    "../vars/session_data_all_MM012_MM013_all5.npz", allow_pickle=True
)["arr_0"]
unit_spike_times_all = np.load(
    "../vars/unit_spike_times_all_MM012_MM013_all5.npz", allow_pickle=True
)["arr_0"]
regions_all = np.load("../vars/regions_all_MM012_MM013_all5.npz", allow_pickle=True)[
    "arr_0"
]

subj_idx = 0
sess_idx = 5

unit_spike_times = unit_spike_times_all[subj_idx][sess_idx]
trial_data = trial_data_all[subj_idx][sess_idx]
session_data = session_data_all[subj_idx][sess_idx]
regions = regions_all[subj_idx][sess_idx]
trial_data["block_side"] = np.where(trial_data["block_side"] == "left", 1, -1)

In [ ]:
family = LVMFamily(
    trial_data=trial_data,
    spike_times=unit_spike_times,
    session_data=session_data,
    regions=regions,
    n_latents_mult=1,
    n_latents_addt=1,
    sanity_check=2,
    task_vars=["response", "rewarded", "response_prev", "rewarded_prev"],
    refit=False,
    norm_activity=False,
)
family.fit_all()
family.eval()

In [ ]:
renderer = FitRenderer(
    family.mod_offset,
    x=family.test_dl.dataset[:],
    y=family.robs,
    save_subdir=Path("model_fits") / "0312-lm" / "offset",
)

nv = NeuronViewer(num_units=renderer.y.shape[1], render_func=renderer)

In [ ]:
# check if you can get back m2 z-score psths from dividing tuber
# ask joao about the masking of the problem via zscore

In [ ]:
plot_summary(family, model=family.mod_affine, potato=family.block_side, mode="affine")